In [7]:
import os
import glob
import torch
import numpy as np
from PIL import Image
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [8]:
print("CUDA Available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Target Device:", device)

CUDA Available: True
Target Device: cuda


In [9]:
class LandCoverMaskDataset(Dataset):
    def __init__(self, base_dir, transform=None):
        # Look for all image files in the directory path
        self.image_paths = glob.glob(os.path.join(base_dir, "*.*"))
        self.transform = transform
        
        # Color coding specifications: (R, G, B)
        self.color_map = [
            [0, 255, 255],    # 0: Urban land
            [255, 255, 0],    # 1: Agriculture land
            [255, 0, 255],    # 2: Rangeland
            [0, 255, 0],      # 3: Forest land
            [0, 0, 255],      # 4: Water
            [255, 255, 255],  # 5: Barren land
            [0, 0, 0]         # 6: Unknown
        ]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_img = Image.open(img_path).convert("RGB")
        
        if self.transform:
            mask_img = self.transform(mask_img)
            
        # Convert the normalized tensor back to a 0-255 NumPy array to check pixel values
        img_np = (mask_img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        pixels = img_np.reshape(-1, 3)
        
        # Track the most dominant color within the patch
        unique_colors, counts = np.unique(pixels, axis=0, return_counts=True)
        dominant_color = unique_colors[np.argmax(counts)].tolist()
        
        # Check tolerance limits for compression artifacts
        label_idx = 6  # Default fallback: Unknown
        for i, color in enumerate(self.color_map):
            if all(abs(dominant_color[j] - color[j]) < 15 for j in range(3)):
                label_idx = i
                break
                
        return mask_img, torch.tensor(label_idx, dtype=torch.long)

In [10]:
# Image preprocessing pipeline
transform = transforms.Compose([
    transforms.Resize((64, 64)),  
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Initialize DataLoaders
train_data = LandCoverMaskDataset("dataset_2/train", transform=transform)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

test_data = LandCoverMaskDataset("dataset_2/test", transform=transform)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print(f"Loaded {len(train_data)} training samples.")
print(f"Loaded {len(test_data)} testing samples.")

Loaded 1606 training samples.
Loaded 172 testing samples.


In [11]:
# Instantiate model with ImageNet pre-trained feature extraction layers
model = models.resnet18(weights='DEFAULT')
model.fc = nn.Linear(model.fc.in_features, 7) 
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
epochs = 6

print(model.fc)  # Confirm final layer size is Out_Features=7

Linear(in_features=512, out_features=7, bias=True)


In [13]:
for epoch in range(epochs):
    model.train()
    total_loss = 0 

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} -> Training Loss Average: {total_loss/len(train_loader):.4f}")
    
    # Run evaluation cycle against test assets
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for test_images, test_labels in test_loader:
            test_images = test_images.to(device)
            test_labels = test_labels.to(device)
            
            test_outputs = model(test_images)
            predictions = torch.argmax(test_outputs, 1)
            
            total += test_labels.size(0)
            correct += (predictions == test_labels).sum().item()
            
    accuracy = (correct / total) * 100 if total > 0 else 0
    print(f"Epoch {epoch+1} Evaluation Accuracy Result: {accuracy:.2f}%\n" + "-"*50)


Epoch 1/6 -> Training Loss Average: 0.3952
Epoch 1 Evaluation Accuracy Result: 100.00%
--------------------------------------------------
Epoch 2/6 -> Training Loss Average: 0.0460
Epoch 2 Evaluation Accuracy Result: 100.00%
--------------------------------------------------
Epoch 3/6 -> Training Loss Average: 0.0211
Epoch 3 Evaluation Accuracy Result: 100.00%
--------------------------------------------------
Epoch 4/6 -> Training Loss Average: 0.0129
Epoch 4 Evaluation Accuracy Result: 100.00%
--------------------------------------------------
Epoch 5/6 -> Training Loss Average: 0.0086
Epoch 5 Evaluation Accuracy Result: 100.00%
--------------------------------------------------
Epoch 6/6 -> Training Loss Average: 0.0065
Epoch 6 Evaluation Accuracy Result: 100.00%
--------------------------------------------------


In [14]:
# Export runtime state parameters
torch.save(model.state_dict(), "torch_wmask_model.pth")
print("Model saved to disk successfully!")

Model saved to disk successfully!
